# Clasificar mineral y estéril con regresión logística
### Taller 01 · Python aplicado a minería — Nilson R. Garrido

En la **ley de corte**, cada muestra de un sondaje se clasifica como **mineral** (va a planta) o **estéril** (va al botadero). Equivocarse tiene dos costos opuestos: enviar estéril a planta **diluye** la ley alimentada, y botar mineral bueno es **pérdida** directa de metal.

En este taller entrenamos una **regresión logística** que reproduce esa clasificación a partir de dos mediciones geoquímicas rápidas (estilo *pXRF*): la **ley de Cu (%)** y el **Mo (ppm)**, ambas elevadas en zonas mineralizadas de un pórfido de Cu-Mo. El flujo es reproducible de principio a fin.

> **Stack:** Python · NumPy · pandas · Matplotlib · scikit-learn · SciPy

## 1) Contexto: la decisión mineral / estéril

El control de leyes (*ore control*) decide, banco a banco y muestra a muestra, el destino de la roca. La regla clásica es un **corte por ley**: si la ley supera la ley de corte, es mineral. Pero la ley de laboratorio tarda horas o días; en el frente se necesitan decisiones rápidas.

Una alternativa es **predecir la clasificación del geólogo** a partir de mediciones inmediatas de campo (un lector portátil de fluorescencia de rayos X, *pXRF*, entrega Cu y Mo en segundos). Si un modelo reproduce con confianza la clasificación experta desde esas dos variables, se gana velocidad sin perder criterio.

Este es un problema de **clasificación binaria**, y la regresión logística es la herramienta canónica: interpretable, calibrada y sencilla de auditar.

## 2) Marco teórico

### 2.1) Regresión logística

La regresión logística modela la **probabilidad** de que una muestra pertenezca a la clase positiva (mineral) como una función *sigmoide* de una combinación lineal de las variables:

$$ p(\text{mineral}) = \sigma(z) = \frac{1}{1 + e^{-z}}, \qquad z = \beta_0 + \beta_1 x_1 + \beta_2 x_2 $$

El término $z$ es el **log-odds** (logaritmo de la razón de momios):

$$ \ln\!\left(\frac{p}{1-p}\right) = \beta_0 + \beta_1\,\text{Cu} + \beta_2\,\text{Mo} $$

A diferencia de la regresión lineal (que estima un valor continuo), la logística estima una probabilidad entre 0 y 1. Se clasifica como mineral cuando $p \geq$ un **umbral** (por defecto 0.5). El umbral no es sagrado: se ajusta según el costo de cada error (Sección 8).

### 2.2) Interpretación de los coeficientes

Con las variables **estandarizadas**, el signo y la magnitud de cada $\beta$ dicen cuánto y en qué dirección pesa cada variable. Un $\beta$ positivo grande para el Cu significa que **más cobre empuja la clasificación hacia mineral**. Comparar $|\beta_{Cu}|$ y $|\beta_{Mo}|$ revela cuál variable domina la decisión.

### 2.3) Geoquímica: por qué Cu y Mo

En los **pórfidos de cobre**, el molibdeno (Mo) acompaña al cobre como subproducto: ambos se concentran en el núcleo mineralizado y decaen hacia la periferia estéril. Por eso Cu y Mo **co-varían** positivamente y, juntos, separan mejor mineral de estéril que cualquiera por sí solo.

## 3) Datos

### 3.1) Variables

| Variable | Símbolo | Unidad | Rango | Descripción |
|---|---|---|---|---|
| Ley de cobre | Cu | % | 0.09 – 1.90 | Ley de Cu de la muestra (pXRF) |
| Molibdeno | Mo | ppm | 11 – 289 | Contenido de Mo (pathfinder de pórfido) |
| Clase | mineral | 0/1 | — | 1 = mineral, 0 = estéril (etiqueta del geólogo) |

### 3.2) Parámetros del sitio simulado (*ground truth*)

Generamos un dataset sintético físicamente plausible. La etiqueta proviene de un modelo logístico **conocido**, lo que permite validar que el modelo ajustado recupera la dirección verdadera.

| Parámetro | Valor | Justificación |
|---|---|---|
| $\beta_1$ (Cu) | 2.30 | Cu como driver dominante de la mineralización |
| $\beta_2$ (Mo) | 1.15 | Mo como co-indicador, la mitad del peso del Cu |
| $\beta_0$ | −0.15 | Intercepto (más estéril que mineral, típico de una campaña) |
| n | 240 | Muestras de la campaña |
| semilla | 2026 | Reproducibilidad |

La correlación Cu-Mo (~0.5) y la superposición de las dos poblaciones cerca de la ley de corte hacen el problema realista: no se resuelve con un umbral trivial en una sola variable.

## 4) Implementación en Python

### 4.1) Librerías

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (accuracy_score, confusion_matrix, roc_auc_score,
                             roc_curve, precision_score, recall_score, f1_score)
from sklearn.model_selection import cross_val_score, StratifiedKFold

SEMILLA = 2026
rng = np.random.default_rng(SEMILLA)

### 4.2) Generación del dataset

Un factor latente de mineralización correlaciona Cu y Mo; la etiqueta se muestrea del modelo logístico verdadero (`B0, B1, B2`). El ruido de etiqueta reproduce la realidad de que no todas las muestras de alta ley se clasifican como mineral (dilución geológica, contactos, etc.).

In [ ]:
# Ground truth
N = 240
B0, B1, B2 = -0.15, 2.30, 1.15      # logit = B0 + B1*z_cu + B2*z_mo

fac = rng.normal(size=N)            # factor latente de mineralizacion
cu = np.clip(np.exp(-1.00 + 0.52 * (0.80 * fac + 0.62 * rng.normal(size=N))), 0.03, 3.0)   # %
mo = np.clip(np.exp(3.90 + 0.55 * (0.70 * fac + 0.70 * rng.normal(size=N))), 3.0, 800.0)   # ppm

zc = (cu - cu.mean()) / cu.std()
zm = (mo - mo.mean()) / mo.std()
p_true = 1.0 / (1.0 + np.exp(-(B0 + B1 * zc + B2 * zm)))
mineral = (rng.random(N) < p_true).astype(int)   # 1 = mineral, 0 = esteril

df = pd.DataFrame({'cu': np.round(cu, 3), 'mo': np.round(mo, 1), 'mineral': mineral})
df = df.sort_values('cu').reset_index(drop=True)
df.head(8)

In [ ]:
# (opcional) guardar el dataset para el repositorio
import os
os.makedirs('../data/raw', exist_ok=True)
df.to_csv('../data/raw/mineral_esteril.csv', index=False)
print(f'{len(df)} muestras | mineral={df.mineral.sum()} ({df.mineral.mean():.0%}) | esteril={(df.mineral==0).sum()}')

## 5) Análisis exploratorio (EDA)

### 5.1) Estadística descriptiva

In [ ]:
df[['cu', 'mo']].describe().round(2)

### 5.2) Distribuciones por clase

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for ax, col, unidad in zip(axes, ['cu', 'mo'], ['Ley de Cu (%)', 'Mo (ppm)']):
    ax.hist(df[df.mineral == 1][col], bins=20, alpha=0.7, color='#0f766e', label='Mineral', edgecolor='white')
    ax.hist(df[df.mineral == 0][col], bins=20, alpha=0.7, color='#b4312a', label='Estéril', edgecolor='white')
    ax.set_xlabel(unidad); ax.set_ylabel('Frecuencia'); ax.legend()
fig.suptitle('Distribución de Cu y Mo por clase'); plt.tight_layout(); plt.show()

Las dos poblaciones se **solapan** cerca de la ley de corte: hay estéril con algo de Cu y mineral de ley moderada. Ese solape es lo que hace útil combinar ambas variables.

### 5.3) Dispersión Cu–Mo y correlación

In [ ]:
corr = df[['cu', 'mo']].corr().iloc[0, 1]
fig, ax = plt.subplots(figsize=(7.5, 6))
for cls, c, m, lab in [(1, '#065f46', 'o', 'Mineral'), (0, '#b4312a', 's', 'Estéril')]:
    d = df[df.mineral == cls]
    ax.scatter(d.cu, d.mo, c=c, marker=m, s=28, alpha=0.75, edgecolors='white', linewidths=0.4, label=lab)
ax.set_xlabel('Ley de Cu (%)'); ax.set_ylabel('Mo (ppm)')
ax.set_title(f'Mineral vs estéril en el espacio Cu–Mo  (correlación Cu-Mo = {corr:.2f})')
ax.legend(); plt.tight_layout(); plt.show()

El mineral ocupa la esquina de **alto Cu y alto Mo**; el estéril, la de bajo Cu y bajo Mo. La frontera entre ambos es lo que el modelo debe aprender.

## 6) Ajuste del modelo

### 6.1) Estandarizar y entrenar

Se estandarizan las variables (media 0, desviación 1) para que Cu y Mo, con escalas muy distintas (% vs ppm), pesen en igualdad y sus coeficientes sean comparables.

In [ ]:
X = df[['cu', 'mo']].values
y = df['mineral'].values

scaler = StandardScaler().fit(X)
Xs = scaler.transform(X)

modelo = LogisticRegression().fit(Xs, y)

coef = modelo.coef_[0]
print(f'Coeficiente Cu: {coef[0]:+.3f}   (ground truth {B1:+.2f})')
print(f'Coeficiente Mo: {coef[1]:+.3f}   (ground truth {B2:+.2f})')
print(f'Intercepto:     {modelo.intercept_[0]:+.3f}   (ground truth {B0:+.2f})')
print(f'Accuracy (in-sample): {accuracy_score(y, modelo.predict(Xs)):.3f}')

El modelo **recupera la dirección verdadera**: ambos coeficientes son positivos y el de **Cu domina** al de Mo (~3× su magnitud), consistente con el *ground truth* del pórfido. Que las magnitudes no coincidan exactamente con los valores verdaderos es esperable: la regularización de scikit-learn (`C=1`) y el ruido de etiqueta introducen sesgo y varianza.

### 6.2) La frontera de decisión

In [ ]:
xx, yy = np.meshgrid(np.linspace(df.cu.min() - 0.05, df.cu.max() + 0.05, 300),
                     np.linspace(df.mo.min() - 10, df.mo.max() + 10, 300))
grid = scaler.transform(np.c_[xx.ravel(), yy.ravel()])
Z = modelo.predict_proba(grid)[:, 1].reshape(xx.shape)

fig, ax = plt.subplots(figsize=(7.5, 6))
ax.contourf(xx, yy, Z, levels=[0, 0.5, 1], colors=['#fdeeee', '#eafaf3'])
ax.contour(xx, yy, Z, levels=[0.5], colors=['#0f766e'], linewidths=2)
for cls, c, m, lab in [(1, '#065f46', 'o', 'Mineral'), (0, '#b4312a', 's', 'Estéril')]:
    d = df[df.mineral == cls]
    ax.scatter(d.cu, d.mo, c=c, marker=m, s=26, alpha=0.8, edgecolors='white', linewidths=0.4, label=lab)
ax.set_xlabel('Ley de Cu (%)'); ax.set_ylabel('Mo (ppm)')
ax.set_title('Frontera de decisión aprendida (probabilidad = 0.5)')
ax.legend(); plt.tight_layout(); plt.show()

## 7) Validación

### 7.1) Matriz de confusión

Cruza lo real contra lo predicho. En minería los dos errores tienen precios distintos: un **falso positivo** (estéril clasificado mineral) va a planta y **diluye**; un **falso negativo** (mineral clasificado estéril) se bota y es **pérdida**.

In [ ]:
pred = modelo.predict(Xs)
cm = confusion_matrix(y, pred)   # [[TN, FP], [FN, TP]]

fig, ax = plt.subplots(figsize=(5.6, 5))
ax.imshow(cm, cmap='Greens', vmin=0, vmax=cm.max())
labs = ['Estéril', 'Mineral']; notas = [['OK', 'dilución'], ['pérdida', 'OK']]
ax.set_xticks([0, 1], labels=labs); ax.set_yticks([0, 1], labels=labs)
ax.set_xlabel('Predicho por el modelo'); ax.set_ylabel('Real (geólogo)')
for i in range(2):
    for j in range(2):
        ax.text(j, i - 0.08, cm[i, j], ha='center', fontsize=18, fontweight='bold',
                color='white' if cm[i, j] > cm.max() * 0.5 else '#16241d')
        ax.text(j, i + 0.22, notas[i][j], ha='center', fontsize=10, style='italic',
                color='white' if cm[i, j] > cm.max() * 0.5 else '#5f6f67')
plt.title('Matriz de confusión'); plt.tight_layout(); plt.show()

print(f'Falsos positivos (dilución): {cm[0,1]}   Falsos negativos (pérdida): {cm[1,0]}')
print(f'Precisión: {precision_score(y, pred):.3f}   Recall: {recall_score(y, pred):.3f}   F1: {f1_score(y, pred):.3f}')

### 7.2) Curva ROC y AUC

In [ ]:
proba = modelo.predict_proba(Xs)[:, 1]
fpr, tpr, _ = roc_curve(y, proba)
auc = roc_auc_score(y, proba)

fig, ax = plt.subplots(figsize=(6, 5.5))
ax.plot(fpr, tpr, color='#0f766e', lw=2.5, label=f'Modelo (AUC = {auc:.3f})')
ax.plot([0, 1], [0, 1], '--', color='#94a3a0', label='Azar (AUC = 0.5)')
ax.set_xlabel('Tasa de falsos positivos'); ax.set_ylabel('Tasa de verdaderos positivos')
ax.set_title('Curva ROC'); ax.legend(loc='lower right'); plt.tight_layout(); plt.show()

### 7.3) Validación cruzada

El AUC in-sample puede ser optimista. La validación cruzada estratificada estima el desempeño en datos no vistos.

In [ ]:
cv = cross_val_score(LogisticRegression(), Xs, y,
                     cv=StratifiedKFold(5, shuffle=True, random_state=SEMILLA), scoring='roc_auc')
print(f'AUC por fold: {np.round(cv, 3)}')
print(f'AUC 5-fold CV: {cv.mean():.3f} +/- {cv.std():.3f}')

## 8) Uso operativo: el umbral y el costo del error

El umbral por defecto (0.5) equilibra ambos errores por igual. Pero en operación el costo de diluir y el de perder mineral rara vez son iguales. Barremos el umbral y contamos cada tipo de error.

In [ ]:
umbrales = np.linspace(0.15, 0.85, 15)
fp = np.array([int(((y == 0) & (proba >= t)).sum()) for t in umbrales])   # dilución
fn = np.array([int(((y == 1) & (proba < t)).sum()) for t in umbrales])    # pérdida

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(umbrales, fp, color='#b4312a', lw=2, marker='o', ms=4, label='Estéril a planta (dilución)')
ax.plot(umbrales, fn, color='#0f766e', lw=2, marker='s', ms=4, label='Mineral botado (pérdida)')
ax.axvline(0.5, ls='--', color='#94a3a0'); ax.set_xlabel("Umbral para declarar 'mineral'")
ax.set_ylabel('N° de errores'); ax.set_title('Compensación dilución vs pérdida'); ax.legend()
plt.tight_layout(); plt.show()

### 8.1) Elegir el umbral por costo

Si asignamos un costo relativo a cada error, el umbral óptimo es el que **minimiza el costo total** $C = c_{dil}\cdot FP + c_{perd}\cdot FN$. Por ejemplo, si perder mineral cuesta el doble que diluir:

In [ ]:
c_dil, c_perd = 1.0, 2.0    # perder mineral cuesta el doble que diluir
costo = c_dil * fp + c_perd * fn
i_opt = int(np.argmin(costo))
tabla = pd.DataFrame({'umbral': umbrales.round(2), 'dilucion_FP': fp, 'perdida_FN': fn,
                      'costo_total': costo.round(1)})
print(tabla.to_string(index=False))
print(f'\nUmbral óptimo para c_perdida/c_dilucion = {c_perd/c_dil:.0f}: {umbrales[i_opt]:.2f}')

El umbral óptimo se **corre** según la relación de costos: si la dilución es cara, sube; si perder metal duele más, baja. El modelo aporta la evidencia cuantitativa; la elección del punto es una **decisión de ingeniería**.

## 9) Conclusiones

- Una regresión logística con dos variables geoquímicas (**Cu y Mo**) reproduce la clasificación mineral/estéril con **AUC ≈ 0.93** y **~89 % de acierto**, validado por validación cruzada.
- El modelo **recupera la dirección del *ground truth***: el Cu domina la decisión y el Mo aporta como co-indicador, consistente con la geoquímica de un pórfido de Cu-Mo.
- La **matriz de confusión** separa los dos errores mineros: falsos positivos = dilución, falsos negativos = pérdida. No cuestan igual.
- El **umbral** no es fijo: se elige minimizando el costo total según la relación dilución/pérdida del proyecto. La técnica aporta la evidencia; el criterio minero toma la decisión.

Este flujo, con mediciones rápidas de campo (pXRF), habilita control de leyes en el frente sin esperar el laboratorio, siempre bajo la supervisión del geólogo.

## 10) Referencias

- Hosmer, D. W., Lemeshow, S., & Sturdivant, R. X. (2013). *Applied Logistic Regression* (3rd ed.). Wiley.
- Lane, K. F. (1988). *The Economic Definition of Ore: Cut-off Grades in Theory and Practice*. Mining Journal Books.
- Sinclair, A. J., & Blackwell, G. H. (2002). *Applied Mineral Inventory Estimation*. Cambridge University Press.
- Sillitoe, R. H. (2010). Porphyry Copper Systems. *Economic Geology*, 105(1), 3–41.
- Pedregosa, F. et al. (2011). Scikit-learn: Machine Learning in Python. *JMLR*, 12, 2825–2830.
- Fawcett, T. (2006). An introduction to ROC analysis. *Pattern Recognition Letters*, 27(8), 861–874.

---
*Nilson Rolando Garrido Asenjo · Ingeniero de Minas · Python para Minería · [linkedin.com/in/nrgarridoa](https://www.linkedin.com/in/nrgarridoa) · [nrgarridoa.github.io](https://nrgarridoa.github.io)*